# GhostCapture AI · Colab

Select a GPU runtime and upload your source ZIP and videos to the session. Set the paths in the configuration cell and run all cells. This uploads or attaches footage to Colab. Use the desktop CLI instead to keep footage on your own machine.

The neural model learns from unmasked areas of each recording and writes its own weights. Multiple compatible CUDA devices each process separate video chunks. A single NVIDIA T4 uses the same code path. Accelerator availability depends on provider quotas.


In [ ]:
from pathlib import Path

SOURCE_ZIP = Path("/content/ghostcapture-source/GhostCapture-AI.zip")
VIDEO_DIR = Path("/content/ghostcapture-videos")
PROFILE = None  # Path to a GhostCapture profile JSON. Use PROFILE or MASK for known layouts.
MASK = None     # Path to a white-on-black mask PNG.
WORK = Path("/content/ghostcapture-work")
CHECKPOINTS = Path("/content/ghostcapture-checkpoints")
RESULTS = Path("/content/ghostcapture-exports")
RESTORE_FROM = None  # Path to a previously downloaded checkpoint ZIP, if resuming a new session.
NEURAL_STEPS = 150
CHUNK_FRAMES = 120
CRF = 18
CPU_FALLBACK = False
CLEAN_AFTER_EXPORT = False  # True removes completed checkpoints and intermediates.


In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import zipfile

def unpack(archive, destination):
    destination = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as package:
        for item in package.infolist():
            if not (destination / item.filename).resolve().is_relative_to(destination):
                raise ValueError(f"Unsafe archive member: {item.filename}")
        package.extractall(destination)

unpack(SOURCE_ZIP, WORK)
entries = sorted(WORK.rglob("desktop/ghostcapture.py"))
if len(entries) != 1:
    raise ValueError("Use the GhostCapture source archive and a fresh WORK directory.")
CLI = entries[0]
if any(not shutil.which(tool) for tool in ("ffmpeg", "ffprobe")):
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CLI.parent / "requirements.txt")], check=True)
if NEURAL_STEPS and not importlib.util.find_spec("torch"):
    subprocess.run([sys.executable, "-m", "pip", "install", "torch>=2.6,<3"], check=True)
if RESTORE_FROM is not None:
    unpack(RESTORE_FROM, CHECKPOINTS)

DEVICE = "cpu"
if NEURAL_STEPS:
    import torch
    if torch.cuda.is_available():
        usable = []
        for index in range(torch.cuda.device_count()):
            device = torch.device(f"cuda:{index}")
            torch.zeros(1, device=device).add_(1).cpu()
            print(f"{device}: {torch.cuda.get_device_name(index)}, CUDA {torch.version.cuda}")
            usable.append(device)
        if usable:
            DEVICE = "cuda"
    if DEVICE == "cpu" and not CPU_FALLBACK:
        raise RuntimeError("No CUDA GPU is available. Enable a GPU accelerator or set CPU_FALLBACK=True.")


In [ ]:
VIDEOS = sorted(path for path in VIDEO_DIR.iterdir()
                if path.is_file() and path.suffix.lower() in (".mp4", ".mov", ".mkv", ".webm"))
if not VIDEOS:
    raise FileNotFoundError(f"No supported video files in {VIDEO_DIR}")
if PROFILE and MASK:
    raise ValueError("Set either PROFILE or MASK.")
MASK_ARGS = ["--profile", str(PROFILE)] if PROFILE else ["--mask", str(MASK)] if MASK else []
PREVIEW = RESULTS / "previews"
for video in VIDEOS:
    subprocess.run([sys.executable, str(CLI), str(video), *MASK_ARGS, "--preview-only",
                    "--output-dir", str(PREVIEW), "--checkpoints", str(CHECKPOINTS), "--overwrite"], check=True)
from IPython.display import Image, display
for file in sorted(PREVIEW.glob("*-mask.png")):
    print(file.name)
    display(Image(filename=str(file)))


Inspect the mask previews before running the export. The automatic heuristic favors stationary, high-contrast overlays and may miss or misidentify HUD. For a customized HUD, supply a profile or PNG mask. Reconstructed pixels estimate the hidden image.

In [ ]:
command = [sys.executable, str(CLI), *map(str, VIDEOS), *MASK_ARGS,
           "--output-dir", str(RESULTS), "--checkpoints", str(CHECKPOINTS),
           "--chunk-frames", str(CHUNK_FRAMES), "--crf", str(CRF),
           "--neural-steps", str(NEURAL_STEPS), "--device", DEVICE, "--overwrite"]
if CLEAN_AFTER_EXPORT:
    command.append("--clean-checkpoints")
subprocess.run(command, check=True)
for file in sorted(RESULTS.glob("*.mp4")):
    print(file)


A completed chunk is reused if the input bytes and processing options match. Session storage is temporary. To resume after a reset, archive and download your checkpoints while the session is still accessible. An export or notebook output saved on the service can contain portions of your original gameplay.

In [ ]:
SAVE_CHECKPOINT_ARCHIVE = False
if SAVE_CHECKPOINT_ARCHIVE:
    archive = shutil.make_archive(str(CHECKPOINTS.with_name("ghostcapture-resume")), "zip", CHECKPOINTS)
    print(f"Download {archive} before ending the session.")
